In [1]:
# Cell 1: Basic imports
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

import google.generativeai as genai

# Configure Gemini
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

print("Gemini API key loaded:", os.getenv("GEMINI_API_KEY")[:8], "...")


Gemini API key loaded: AIzaSyCx ...


In [3]:
# Loading the documents 
from pypdf import PdfReader
from pathlib import Path

PDF_PATH = Path("../datas/ipc.pdf")   # adjust path if needed

def load_pdf_text(path):
    reader = PdfReader(str(path))
    pages = []
    for p in reader.pages:
        txt = p.extract_text() or ""
        pages.append(txt)
    return "\n".join(pages)

raw_text = load_pdf_text(PDF_PATH)

print("Characters extracted:", len(raw_text))
print(raw_text[:500])


Characters extracted: 4481831
THE INDIAN PENAL CODE
CHAPTER I INTRODUCTION
The Indian Penal Code was drafted by the First Indian Law Commission presided over
by Lord Thomas Babington Macaulay. The draft underwent further revision at the hands
of well-known jurists, like Sir Barnes Peacock, and was completed in 1850. The Indian
Penal Code was passed by the then Legislature on 6 October 1860 and was enacted as
Act No. XLV of 1860.
Preamble. WHEREAS it is expedient to provide a general
Penal Code for India; It is enacted as fol


In [4]:
# Creating chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_text(raw_text)
print("Total chunks:", len(chunks))


Total chunks: 6776


In [5]:
#converting chunks into embeddings
def embed_text(text):
    resp = genai.embed_content(
        model="text-embedding-004",
        content=text
    )
    return resp["embedding"]

# test single embedding
print(len(embed_text("hello")), "dims")


768 dims


In [8]:
# Cell C: FAISS fallback — builds and persist index (faiss-cpu required)
# Install faiss-cpu if not installed: !pip install faiss-cpu
import numpy as np, faiss, pickle
from pathlib import Path
from tqdm.auto import tqdm

# make sure 'chunks' exists in memory; if not, raise friendly error
try:
    _ = chunks  # noqa
except NameError:
    raise RuntimeError("FAISS fallback needs 'chunks' variable in your notebook. Make sure you have created 'chunks' earlier.")

PERSIST_DIR = Path("./faiss_store")
PERSIST_DIR.mkdir(exist_ok=True)
INDEX_PATH = PERSIST_DIR / "faiss.index"
META_PATH  = PERSIST_DIR / "metadata.pkl"

# compute embeddings for chunks (reuses 'embs' if present)
try:
    emb_array = np.array(embs, dtype=np.float32)
    print("Using existing 'embs' variable. shape:", emb_array.shape)
except Exception:
    print("Computing embeddings for chunks (calls embed_text for each chunk).")
    vecs = []
    for ch in tqdm(chunks, desc="embedding"):
        vecs.append(np.array(embed_text(ch), dtype=np.float32))
    emb_array = np.vstack(vecs)

# normalize (for cosine using IP)
norms = np.linalg.norm(emb_array, axis=1, keepdims=True)
norms[norms == 0] = 1.0
emb_array = emb_array / norms

d = emb_array.shape[1]
index = faiss.IndexFlatIP(d)
index.add(emb_array)
faiss.write_index(index, str(INDEX_PATH))

# metadata
meta = {"docs": chunks}
with open(META_PATH, "wb") as f:
    pickle.dump(meta, f)

print("FAISS built & saved:", INDEX_PATH, " docs:", len(chunks))


Computing embeddings for chunks (calls embed_text for each chunk).


embedding:   0%|          | 0/6776 [00:00<?, ?it/s]

FAISS built & saved: faiss_store\faiss.index  docs: 6776


In [9]:
import faiss, pickle, numpy as np

# load index and metadata
index = faiss.read_index("faiss_store/faiss.index")
with open("faiss_store/metadata.pkl", "rb") as f:
    meta = pickle.load(f)

docs = meta["docs"]

def retrieve(query, k=3):
    qvec = np.array(embed_text(query), dtype=np.float32)
    qvec = qvec / (np.linalg.norm(qvec) + 1e-12)
    D, I = index.search(qvec.reshape(1, -1), k)
    return [docs[j] for j in I[0]]

print("Top chunk:\n")
print(retrieve("what is cheating?", k=1)[0][:500])


Top chunk:

[One year, or ﬁne, or both (section 416).]
Like 'extortion', cheating is committed by the wrongful obtaining of a consent. The
difference is that, in the former the consent is obtained by intimidation, in the latter, by
deception.
'Cheating' also differs from 'theft'
(1) In the former the property obtained by deception may be movable or immovable, in
the latter, it must be movable.
(2) In 'theft' the property is taken without the consent of the owner, in 'cheating' the
owner's consent is obtaine


In [11]:
# Cell A — list available models and what they support
import google.generativeai as genai
from pprint import pprint

# Make sure you configured genai earlier: genai.configure(api_key=...)
models = genai.list_models()   # returns a list/dict of models
# print compact info
for m in models:
    name = m.get("name") if isinstance(m, dict) else getattr(m, "name", str(m))
    caps = m.get("capabilities", []) if isinstance(m, dict) else None
    print("MODEL:", name)
    if caps:
        print("  capabilities:", caps)
    else:
        # best-effort fields
        for k in ("availability", "supported_methods", "features"):
            if isinstance(m, dict) and k in m:
                print(f"  {k}:", m[k])
    print()


MODEL: models/embedding-gecko-001

MODEL: models/gemini-2.5-pro-preview-03-25

MODEL: models/gemini-2.5-flash-preview-05-20

MODEL: models/gemini-2.5-flash

MODEL: models/gemini-2.5-flash-lite-preview-06-17

MODEL: models/gemini-2.5-pro-preview-05-06

MODEL: models/gemini-2.5-pro-preview-06-05

MODEL: models/gemini-2.5-pro

MODEL: models/gemini-2.0-flash-exp

MODEL: models/gemini-2.0-flash

MODEL: models/gemini-2.0-flash-001

MODEL: models/gemini-2.0-flash-exp-image-generation

MODEL: models/gemini-2.0-flash-lite-001

MODEL: models/gemini-2.0-flash-lite

MODEL: models/gemini-2.0-flash-lite-preview-02-05

MODEL: models/gemini-2.0-flash-lite-preview

MODEL: models/gemini-2.0-pro-exp

MODEL: models/gemini-2.0-pro-exp-02-05

MODEL: models/gemini-exp-1206

MODEL: models/gemini-2.0-flash-thinking-exp-01-21

MODEL: models/gemini-2.0-flash-thinking-exp

MODEL: models/gemini-2.0-flash-thinking-exp-1219

MODEL: models/gemini-2.5-flash-preview-tts

MODEL: models/gemini-2.5-pro-preview-tts

MODEL:

In [12]:
# Cell B — test a chosen model and inspect the raw response
MODEL_NAME = "gemini-2.5-flash"  # <- change this to a name you saw in Cell A

prompt = "Say hello in one sentence."

try:
    model = genai.GenerativeModel(MODEL_NAME)
    resp = model.generate_content(prompt)
    print("RESPONSE (repr):", repr(resp)[:1000])
    # show attributes to inspect structure
    print("\nAttributes:", [a for a in dir(resp) if not a.startswith("_")][:200])
    # try a few common extraction points safely:
    text_candidates = []
    # candidate common shapes below, try each one if present
    try:
        # newer objects may have .candidates[0].content[0].text
        c = resp.candidates
        print("Found resp.candidates (len):", len(c))
        try:
            # try extracting nested contents
            text_candidates.append(c[0].content[0].text)
        except Exception:
            # fallback: candidate may have .output_text or .text
            if hasattr(c[0], "output_text"):
                text_candidates.append(c[0].output_text)
            elif hasattr(c[0], "text"):
                text_candidates.append(c[0].text)
    except Exception:
        pass

    # older shaped responses might be dict-like
    try:
        if isinstance(resp, dict):
            if "candidates" in resp:
                cc = resp["candidates"]
                if cc and isinstance(cc[0], dict):
                    # try a couple of keys
                    for k in ("content", "text", "output_text", "generated_text"):
                        if k in cc[0]:
                            text_candidates.append(cc[0][k])
    except Exception:
        pass

    print("\nPossible extracted texts (first non-empty):")
    for t in text_candidates:
        if t:
            print("-", t[:400])
            break
    else:
        print("No text extracted automatically — inspect the printed repr above to find where the generated text is stored.")
except Exception as e:
    print("Model call failed:", type(e), e)



RESPONSE (repr): response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "Hello!"
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 7,
        "candidates_token_count": 2,
        "total_token_count": 38
      },
      "model_version": "gemini-2.5-flash"
    }),
)

Attributes: ['candidates', 'from_iterator', 'from_response', 'model_version', 'parts', 'prompt_feedback', 'resolve', 'text', 'to_dict', 'usage_metadata']
Found resp.candidates (len): 1

Possible extracted texts (first non-empty):
No text extracted automatically — inspect the printed repr above to find where the generated text is stored.


In [16]:
def _extract_text_from_genai_response(resp):
    """
    Try several common response shapes and return the first text found:
      - resp.candidates[0].content.parts[0].text
      - resp.candidates[0].text / .output_text
      - dict-like fallbacks (generated_text, output_text, text)
    """
    # 1) object with candidates -> content -> parts -> text
    try:
        if hasattr(resp, "candidates"):
            cand0 = resp.candidates[0]
            # nested content.parts[0].text
            if hasattr(cand0, "content"):
                content = cand0.content
                # content may have .parts (list-like) or be a list
                if hasattr(content, "parts") and content.parts:
                    part0 = content.parts[0]
                    if hasattr(part0, "text"):
                        return part0.text
                # fallback: content might be a list/dict-like
                try:
                    # try indexing & dict style
                    if isinstance(content, (list, tuple)) and content:
                        if isinstance(content[0], dict) and "text" in content[0]:
                            return content[0]["text"]
                except Exception:
                    pass
            # other candidate shapes
            for attr in ("text", "output_text", "generated_text"):
                if hasattr(cand0, attr):
                    return getattr(cand0, attr)
    except Exception:
        pass

    # 2) dict-like response
    try:
        if isinstance(resp, dict):
            # common keys
            for k in ("generated_text", "output_text", "text"):
                if k in resp and isinstance(resp[k], str):
                    return resp[k]
            # candidates array in dict form
            if "candidates" in resp and resp["candidates"]:
                c0 = resp["candidates"][0]
                if isinstance(c0, dict):
                    # try content.parts
                    cont = c0.get("content")
                    if isinstance(cont, dict) and "parts" in cont and cont["parts"]:
                        p0 = cont["parts"][0]
                        if isinstance(p0, dict) and "text" in p0:
                            return p0["text"]
                    for key in ("text", "output_text", "generated_text"):
                        if key in c0 and isinstance(c0[key], str):
                            return c0[key]
    except Exception:
        pass

    # 3) fallback: stringified response
    try:
        return str(resp)
    except Exception:
        return ""


In [18]:
import google.generativeai as genai

WORKING_MODEL = "gemini-2.5-flash"   # you confirmed this works

def rag_answer(question, context_chunks, model_name=WORKING_MODEL):
    """
    context_chunks: list[str] — top-k retrieved chunks
    """
    context = "\n\n".join(context_chunks)
    prompt = f"""You are a legal assistant. Use ONLY the context below to answer the question concisely and cite sections if possible.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
    model = genai.GenerativeModel(model_name)
    resp = model.generate_content(prompt)
    return _extract_text_from_genai_response(resp)

# Example usage:
top_chunks = retrieve("What is cheating under IPC?", k=3)  # your retrieve function
print(rag_answer("Explain cheating under IPC in simple terms.", top_chunks))



Cheating occurs when someone, by deceiving another person, fraudulently or dishonestly induces that person to deliver any property, consent to its retention, or intentionally do or omit to do something they would not otherwise, which causes or is likely to cause damage or harm to that person's body, mind, reputation, or property. A dishonest concealment of facts is considered deception. (s 415)
